# Warp Drive Sky-Flight (Colab GPU, Drive-checkpointed)

Render high-quality videos of what the night sky looks like through
the front window of a spacecraft as it accelerates from rest into
superluminal velocities inside an Alcubierre / Natário warp bubble.

**Pipeline:**

1. Mount Google Drive — every frame is checkpointed there
2. Clone the WarpDrives repo and install in editable mode
3. Switch JAX to the GPU backend (Colab T4 / A100)
4. Download an equirectangular Milky Way panorama, or use the procedural starfield
5. For each velocity in a smooth ramp 0 → v_max, fire backward null geodesics through the warp metric and **save the PNG to Drive**
6. If the runtime times out, just rerun — the loop skips frames that already exist
7. Assemble the per-frame PNGs into MP4/GIF

Suggested runtime: **GPU (T4 or better)** — set via `Runtime → Change runtime type` before running.

## 1. Mount Google Drive (checkpoint store)

Every intermediate frame and the assembled video go into
`MyDrive/WarpDrives/skyflight/<run_name>/`.  Picking a stable
`RUN_NAME` is what makes the render *resumable*: rerun the notebook
with the same name and only the missing frames are recomputed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DRIVE_ROOT = '/content/drive/MyDrive/WarpDrives'
RUN_NAME   = 'alcubierre_v0_to_4c_384px_honest_2026_04_28'   # honest-physics build (Doppler + horizon mask)
RUN_DIR    = os.path.join(DRIVE_ROOT, 'skyflight', RUN_NAME)
FRAMES_DIR = os.path.join(RUN_DIR, 'frames')
META_PATH  = os.path.join(RUN_DIR, 'meta.json')
VIDEO_DIR  = os.path.join(RUN_DIR, 'video')
for d in (RUN_DIR, FRAMES_DIR, VIDEO_DIR):
    os.makedirs(d, exist_ok=True)
print('Run directory:', RUN_DIR)

## 2. Install dependencies and clone the repo

In [ ]:
# --- Hardened install ---------------------------------------------------
# Colab pre-installs numpy 2.x.  pip install -e . then re-resolves and may
# downgrade numpy because of pinned versions in transitive deps.  Pin
# everything we need in ONE call so the resolver picks a self-consistent
# set, and skip pyvista entirely (the sky-flight doesn't use it; the
# package imports it lazily so the import never fires).
!pip install -q --upgrade pip setuptools wheel
!pip install -q \
    "numpy>=2.0,<2.2" \
    "scipy>=1.13" \
    "matplotlib>=3.8" \
    "jax>=0.4.30" "jaxlib>=0.4.30" \
    "imageio>=2.34" "imageio-ffmpeg>=0.5.1" \
    "pyyaml>=6.0" "click>=8.1" "tqdm>=4.66"

# FORCE_REFRESH=True wipes any previously-extracted source so the
# slim zip on Drive (or the wget'd repo) is freshly unpacked.
# Set False to reuse an existing /content/WarpDrives — faster but
# means a stale notebook can run an older version of the package.
FORCE_REFRESH = True

import os, sys, glob
REPO_DIR = '/content/WarpDrives'

if FORCE_REFRESH and os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}
if not os.path.exists(REPO_DIR):
    drive_candidates = [
        '/content/drive/MyDrive/WarpDrives_slim.zip',
        '/content/drive/MyDrive/WarpDrives/WarpDrives_slim.zip',
    ]
    drive_zip = next((p for p in drive_candidates if os.path.exists(p)), None)
    if drive_zip is not None:
        print(f'Found slim zip on Drive: {drive_zip}')
        !unzip -q -o {drive_zip} -d /content
    else:
        print('No slim zip on Drive — trying public GitHub zip via wget…')
        !rm -f /content/main.zip
        !wget -q https://github.com/mthiel74/WarpDrives/archive/refs/heads/main.zip -O /content/main.zip
        if os.path.exists('/content/main.zip') and os.path.getsize('/content/main.zip') > 1000:
            !unzip -q /content/main.zip -d /content
            !mv /content/WarpDrives-main /content/WarpDrives
        else:
            raise SystemExit(
                'Could not obtain WarpDrives source.  Upload\n'
                '~/Desktop/WarpDrives_slim.zip to MyDrive (top level)\n'
                'via drive.google.com, then re-run this cell.'
            )

%cd /content/WarpDrives
# Install the package WITHOUT pulling its dependency list (we already
# pinned them above) so pip doesn't roll numpy back to satisfy old
# pyvista or similar.
!pip install -q --no-deps -e .
sys.path.insert(0, os.getcwd())

import numpy as np
print(f'numpy {np.__version__} | scipy {__import__("scipy").__version__}')

In [ ]:
import jax
print('JAX:', jax.__version__)
print('Devices:', jax.devices())
print('Default backend:', jax.default_backend())

## What you're looking at

- **Camera**: pinhole, 90° FOV, attached to the bubble centre and pointing along the bubble's direction of motion (+x).
- **Velocity ramp**: cosine-eased from `v=0` to `v=V_MAX` over `N_FRAMES` frames, then mirrored back to 0 in the final video. So you start at rest, accelerate, hold briefly at top speed, decelerate.
- **Direction**: the bubble moves into +x. The Milky Way panorama is mapped to the celestial sphere with the galactic plane roughly across the equator, so initially you're looking *into* the band.
- **Doppler brightness**: honest `I_ν / ν³ = const` (Liouville's theorem). Forward pixels brighten as `f^p` with `p=3` for monochromatic intensity (default) or `p=4` for bolometric thermal flux (the §7 A/B render). Trailing pixels dim. **No fake colour shift** — we'd need spectral data to do colour honestly. **No tonemap** by default — the forward-superluminal field really does saturate in the visible band, that's not a render bug.
- **Front horizon**: rays whose backward null geodesic fails to escape the bubble are rendered black, since they have no causal contact with the celestial sphere. With the default integration budget and bubble shape this only activates for cleanly trapped rays, not for the entire `v > 1` forward cone — getting the latter requires a longer integration or a sharper bubble wall.
- **Doppler brightness**: honest `I_ν / ν³ = const` (Liouville's theorem). Forward pixels brighten as `f^p` with `p=3` for monochromatic intensity (default) or `p=4` for bolometric thermal flux (the §7 A/B render). Trailing pixels dim. **No fake colour shift** — we'd need spectral data to do colour honestly. **No tonemap** by default — the forward-superluminal field really does saturate in the visible band, that's not a render bug.
- **Natário caveat**: Natário's curl-based shift gives the central observer a non-timelike worldline at superluminal v. The comparison cell therefore sweeps only up to v=0.95c.
- **Sparkle / flicker**: was a sampling artefact, not physics. Mitigated by `supersample=2` (4 rays/pixel, averaged) and bilinear interpolation in `make_image_sky`.

## 3. Sky background (Drive-cached download)

The Milky Way panorama is downloaded into Drive once.  If you re-run
the notebook (or share the run with another machine) it just reuses
the cached file.

In [ ]:
import urllib.request
MW_URL  = 'https://cdn.eso.org/images/large/eso0932a.jpg'
MW_PATH = os.path.join(DRIVE_ROOT, 'assets', 'milky_way_panorama.jpg')
os.makedirs(os.path.dirname(MW_PATH), exist_ok=True)
if not os.path.exists(MW_PATH):
    print('Downloading Milky Way panorama (ESO/Brunier) into Drive…')
    urllib.request.urlretrieve(MW_URL, MW_PATH)
print('Sky image at:', MW_PATH, '|', os.path.getsize(MW_PATH) // 1024, 'KB')

In [ ]:
import numpy as np
from warpbubblesim.viz.skybackground import (
    make_image_sky, make_procedural_starfield,
)

USE_REAL_PANORAMA = os.path.exists(MW_PATH)
if USE_REAL_PANORAMA:
    print(f'Sky: ESO Milky Way panorama at {MW_PATH}')
else:
    print('Sky: procedural starfield (panorama not on Drive — skipping the')
    print('     ESO download in cell 8 is the easy way to fix this).')
    print('     Procedural stars twinkle visibly under animation; for HQ')
    print('     videos the panorama is strongly recommended.')

## 4. Render configuration & velocity schedule

The schedule is written to `meta.json` so a resumed run uses the
**identical** velocities — important when checkpoints already exist
on Drive.

In [ ]:
from warpbubblesim.viz.skyrender_jax import (
    JaxRenderConfig, render_frame_jax,
)

# ------------ Quality preset (landscape 16:9) ------------
# Pick PREVIEW for fast iteration, HQ for the published render, ULTRA
# if you have a beefy GPU runtime (A100 / H100 etc.).  All three are
# 16:9 landscape; only the resolution / frame count / supersample /
# integration steps change.

PRESET = 'HQ'   # 'PREVIEW' | 'HQ' | 'ULTRA'

PRESETS = {
    'PREVIEW': dict(width=640,  height=360,  n_frames=30,  n_steps=200, supersample=1),
    'HQ':      dict(width=1280, height=720,  n_frames=120, n_steps=280, supersample=2),
    'ULTRA':   dict(width=1920, height=1080, n_frames=180, n_steps=320, supersample=2),
}
p = PRESETS[PRESET]

WIDTH      = p['width']
HEIGHT     = p['height']
N_FRAMES   = p['n_frames']
FOV_DEG    = 90.0     # horizontal FOV; vertical = atan(tan(fov/2)/aspect)
V_MAX      = 4.0
METRIC     = 'alcubierre'      # 'alcubierre' or 'natario'
BASE_PARAMS = dict(R=1.0, sigma=8.0, shape='tanh') if METRIC == 'alcubierre' \
             else dict(R=1.0, sigma=8.0)

JAX_CFG = JaxRenderConfig(
    width=WIDTH, height=HEIGHT, fov_deg=FOV_DEG,
    n_steps=p['n_steps'], dlam=0.15, chunk_size=8192,
    supersample=p['supersample'],
    enable_doppler=True,
    doppler_intensity_power=3.0,
    doppler_tonemap=False,
    enable_horizon_mask=True,
)

RUN_NAME   = f'alcubierre_v0_to_{int(V_MAX)}c_{WIDTH}x{HEIGHT}_{PRESET}'
RUN_DIR    = os.path.join(DRIVE_ROOT, 'skyflight', RUN_NAME)
FRAMES_DIR = os.path.join(RUN_DIR, 'frames')
META_PATH  = os.path.join(RUN_DIR, 'meta.json')
VIDEO_DIR  = os.path.join(RUN_DIR, 'video')
for d in (RUN_DIR, FRAMES_DIR, VIDEO_DIR):
    os.makedirs(d, exist_ok=True)
print(f'Preset: {PRESET} → {WIDTH}×{HEIGHT} (16:9), {N_FRAMES} frames, supersample={p["supersample"]}')
print(f'Total rays per frame: {WIDTH * HEIGHT * p["supersample"]**2:,}')
print(f'Run dir: {RUN_DIR}')

def make_velocities(n_frames, v_max):
    tt = np.linspace(0.0, 1.0, n_frames)
    ease = 0.5 - 0.5 * np.cos(np.pi * tt)
    return (v_max * ease).tolist()

if os.path.exists(META_PATH):
    meta = json.load(open(META_PATH))
    print('Loaded existing schedule from', META_PATH)
    velocities = meta['velocities']
    if (meta['n_frames'] != N_FRAMES or meta['v_max'] != V_MAX
        or meta['metric'] != METRIC or meta.get('preset') != PRESET):
        raise SystemExit(
            f'Existing run {RUN_NAME} was rendered with different settings;'
            f' bump PRESET or RUN_NAME (currently meta={meta}).'
        )
else:
    velocities = make_velocities(N_FRAMES, V_MAX)
    meta = dict(
        run_name=RUN_NAME, metric=METRIC, base_params=BASE_PARAMS,
        width=WIDTH, height=HEIGHT, fov_deg=FOV_DEG,
        n_frames=N_FRAMES, v_max=V_MAX, velocities=velocities, preset=PRESET,
        jax_cfg=dict(n_steps=JAX_CFG.n_steps, dlam=JAX_CFG.dlam,
                     chunk_size=JAX_CFG.chunk_size,
                     supersample=JAX_CFG.supersample),
    )
    json.dump(meta, open(META_PATH, 'w'), indent=2)
    print('Wrote schedule to', META_PATH)

print(f'{len(velocities)} frames, v ∈ [{velocities[0]:.3f}, {max(velocities):.3f}]')

## 5. Render loop with per-frame Drive checkpointing

Each frame is saved as `frames/frame_<idx>_v<vel>.png` *immediately*
after it's computed.  Reruns of this cell skip frames whose PNG
already exists, so a 12-hour timeout in the middle of a sweep just
costs you the partially-rendered frame, not the run.

In [ ]:
import time, glob
import imageio.v2 as iio

def frame_path(idx):
    return os.path.join(FRAMES_DIR, f'frame_{idx:04d}.png')

def build_sky():
    if USE_REAL_PANORAMA:
        return make_image_sky(MW_PATH, rotation_deg=0.0, gain=1.6)
    return make_procedural_starfield(
        n_stars=20000, seed=2026,
        star_radius_px=1.2,
        fov_scale=np.deg2rad(FOV_DEG) / WIDTH,
        soft_blend_top_k=4,
    )

sky = build_sky()

n_pre  = sum(1 for i in range(N_FRAMES) if os.path.exists(frame_path(i)))
print(f'{n_pre}/{N_FRAMES} frames already on Drive — resuming from frame {n_pre}.')

rendered_this_session = 0
t_session = time.time()
for i, v in enumerate(velocities):
    fp = frame_path(i)
    if os.path.exists(fp):
        continue
    t0 = time.time()
    params = dict(BASE_PARAMS, v0=float(v))
    img = render_frame_jax(METRIC, params, sky, JAX_CFG)
    rgb = np.clip(img * 255, 0, 255).astype(np.uint8)
    tmp = fp + '.tmp'
    iio.imwrite(tmp, rgb)
    os.replace(tmp, fp)
    rendered_this_session += 1
    elapsed = time.time() - t0
    sess_t  = time.time() - t_session
    remaining = N_FRAMES - i - 1
    rate = rendered_this_session / max(sess_t, 1e-6)
    eta = remaining / max(rate, 1e-6)
    print(f'  frame {i+1}/{N_FRAMES} v={v:.3f}  {elapsed:.1f}s  '
          f'(rate {rate:.2f}/s, ETA {eta/60:.1f} min)')

n_post = sum(1 for i in range(N_FRAMES) if os.path.exists(frame_path(i)))
print(f'\nDone: {n_post}/{N_FRAMES} frames on Drive.')

## 6. Assemble MP4 and GIF from disk

The video is read **from Drive**, not from in-memory state, so this
cell works in a fresh runtime as long as the frames are checkpointed.

In [ ]:
import imageio.v2 as iio

frames = []
missing = []
for i in range(N_FRAMES):
    fp = frame_path(i)
    if not os.path.exists(fp):
        missing.append(i)
        continue
    frames.append(iio.imread(fp))
if missing:
    print(f'Warning: {len(missing)} frames missing — rerun cell 5 to fill them.'
          f'  Indices: {missing[:10]}{"..." if len(missing) > 10 else ""}')

# Tail-and-back loop: pause at top speed then ramp down
extended = list(frames) + [frames[-1]] * 6 + list(reversed(frames))

MP4 = os.path.join(VIDEO_DIR, f'{RUN_NAME}.mp4')
GIF = os.path.join(VIDEO_DIR, f'{RUN_NAME}.gif')
iio.mimsave(MP4, extended, fps=24)
try:
    iio.mimsave(GIF, extended, duration=int(1000/12), loop=0)
except TypeError:
    iio.mimsave(GIF, extended, fps=12)
print('Wrote', MP4)
print('Wrote', GIF)

In [ ]:
from IPython.display import Video
Video(MP4, embed=True, width=512)

## 7. A/B render: bolometric (f⁴) Doppler — physically honest 'bloom'

Same velocity schedule, rerendered with `doppler_intensity_power=4`
instead of `3`.  Both exponents are physically defensible — they
differ in the assumed source spectrum:

- **f³** (Liouville's `I_ν / ν³ = const`) — exact for monochromatic
  intensity at any one frequency.  This is what the main render uses.
- **f⁴** (bolometric flux scaling for thermal sources) — counts an
  additional factor of `ω` because the visible *bandwidth* also
  Doppler-shifts.  Appropriate when the source is approximately
  blackbody (most stars are).

Frames go to a separate Drive directory; resumable like the main run.

In [ ]:
import dataclasses

BOL_RUN_NAME = RUN_NAME + '_bolometric'
BOL_RUN_DIR  = os.path.join(DRIVE_ROOT, 'skyflight', BOL_RUN_NAME)
BOL_FRAMES   = os.path.join(BOL_RUN_DIR, 'frames')
BOL_VIDEO    = os.path.join(BOL_RUN_DIR, 'video')
for d in (BOL_RUN_DIR, BOL_FRAMES, BOL_VIDEO):
    os.makedirs(d, exist_ok=True)

BOL_CFG = dataclasses.replace(JAX_CFG, doppler_intensity_power=4.0)

def bol_path(i): return os.path.join(BOL_FRAMES, f'frame_{i:04d}.png')

n_pre = sum(1 for i in range(N_FRAMES) if os.path.exists(bol_path(i)))
print(f'{n_pre}/{N_FRAMES} bolometric frames already on Drive — resuming.')

rendered = 0
t_session = time.time()
for i, v in enumerate(velocities):
    fp = bol_path(i)
    if os.path.exists(fp): continue
    t0 = time.time()
    img = render_frame_jax(METRIC, dict(BASE_PARAMS, v0=float(v)), sky, BOL_CFG)
    rgb = np.clip(img * 255, 0, 255).astype(np.uint8)
    tmp = fp + '.tmp'
    iio.imwrite(tmp, rgb); os.replace(tmp, fp)
    rendered += 1
    sess = time.time() - t_session
    rate = rendered / max(sess, 1e-6); eta = (N_FRAMES - i - 1) / max(rate, 1e-6)
    print(f'  bol {i+1}/{N_FRAMES} v={v:.3f} {time.time()-t0:.1f}s ETA {eta/60:.1f} min')

frames = [iio.imread(bol_path(i)) for i in range(N_FRAMES) if os.path.exists(bol_path(i))]
extended = list(frames) + [frames[-1]] * 6 + list(reversed(frames))
BOL_MP4 = os.path.join(BOL_VIDEO, f'{BOL_RUN_NAME}.mp4')
iio.mimsave(BOL_MP4, extended, fps=24)
print('Wrote', BOL_MP4)
Video(BOL_MP4, embed=True, width=512)

## 8. A/B render: cool-source (T_src = 1500 K) — IR shifts INTO visible

If you instead model the panorama's pixels as **cool** thermal sources
(T_src = 1500 K — warm dust / very late-type stars / red giants), their
rest-frame visible-band emission is faint (peak in IR, ~2 μm), but at
Doppler factors `f ≈ 3-4` the apparent temperature climbs into the
stellar-photosphere range and visible emission *re-emerges*.  This is
the symmetric half of the SR-Doppler picture: while solar-temperature
stars dim into UV, cooler sources brighten as their IR shifts into
visible.

Same panorama, same schedule, only `doppler_T_src` differs.  Compare
this video against §7 to see the source-temperature dependence of
what the windshield actually shows.

In [ ]:
COOL_RUN_NAME = RUN_NAME + '_cool_T1500'
COOL_RUN_DIR  = os.path.join(DRIVE_ROOT, 'skyflight', COOL_RUN_NAME)
COOL_FRAMES   = os.path.join(COOL_RUN_DIR, 'frames')
COOL_VIDEO    = os.path.join(COOL_RUN_DIR, 'video')
for d in (COOL_RUN_DIR, COOL_FRAMES, COOL_VIDEO):
    os.makedirs(d, exist_ok=True)

COOL_CFG = dataclasses.replace(
    JAX_CFG,
    doppler_mode='blackbody',     # spectral-aware Doppler
    doppler_T_src=1500.0,         # cool sources (warm dust / red giants)
    doppler_intensity_power=4.0,  # bolometric for thermal
)

def cool_path(i): return os.path.join(COOL_FRAMES, f'frame_{i:04d}.png')

n_pre = sum(1 for i in range(N_FRAMES) if os.path.exists(cool_path(i)))
print(f'{n_pre}/{N_FRAMES} cool-source frames already on Drive — resuming.')

rendered = 0
t_session = time.time()
for i, v in enumerate(velocities):
    fp = cool_path(i)
    if os.path.exists(fp): continue
    t0 = time.time()
    img = render_frame_jax(METRIC, dict(BASE_PARAMS, v0=float(v)), sky, COOL_CFG)
    rgb = np.clip(img * 255, 0, 255).astype(np.uint8)
    tmp = fp + '.tmp'
    iio.imwrite(tmp, rgb); os.replace(tmp, fp)
    rendered += 1
    sess = time.time() - t_session
    rate = rendered / max(sess, 1e-6); eta = (N_FRAMES - i - 1) / max(rate, 1e-6)
    print(f'  cool {i+1}/{N_FRAMES} v={v:.3f} {time.time()-t0:.1f}s ETA {eta/60:.1f} min')

frames = [iio.imread(cool_path(i)) for i in range(N_FRAMES) if os.path.exists(cool_path(i))]
extended = list(frames) + [frames[-1]] * 6 + list(reversed(frames))
COOL_MP4 = os.path.join(COOL_VIDEO, f'{COOL_RUN_NAME}.mp4')
iio.mimsave(COOL_MP4, extended, fps=24)
print('Wrote', COOL_MP4)
Video(COOL_MP4, embed=True, width=512)

## 9. Multiband sky: layer real IR maps for honest IR→visible shift

The blackbody mode in §8 *infers* the source spectrum (assumes one
T_src for every pixel).  The fully honest version uses **multi-
wavelength sky maps** — one panorama per band — and selects, per ray
and per Doppler factor, the band whose source-frame wavelength range
lands in the observer's visible after shift.

**Where to get IR sky maps** (all freely available; you'll need to
download equirectangular versions and upload them to Drive):

| Band | Wavelength | Source |
|---|---|---|
| Visible (Milky Way panorama) | 380–700 nm | ESO eso0932a (already in cell 8) |
| 2MASS J | 1.14–1.37 µm | https://irsa.ipac.caltech.edu/2MASS/ |
| 2MASS H | 1.49–1.81 µm | (same) |
| 2MASS K | 2.00–2.31 µm | (same) |
| WISE W1 | 3.0–3.8 µm | https://irsa.ipac.caltech.edu/Missions/wise.html |
| WISE W2 | 4.2–5.0 µm | (same) |
| WISE W3 | 8–16 µm | (same) |
| IRAS 12/25/60/100 µm | 8–120 µm | https://irsa.ipac.caltech.edu/IRASdocs/ |
| Planck 100/143/217 GHz | 1.4–3 mm | http://pla.esac.esa.int/ |

Drop them in `MyDrive/WarpDrives/sky/` with the names below, or edit
the dict to point at wherever you put them.

If a band's file is missing the code falls back to a **synthetic IR
panorama** generated procedurally — populated with a different
starfield seed plus a thicker dust band, so the renderer at least
*does something* multiband-like for visualisation purposes.

In [ ]:
from warpbubblesim.viz.skybackground import (
    make_image_sky, make_procedural_starfield, make_multiband_sky,
)

SKY_DIR = '/content/drive/MyDrive/WarpDrives/sky'
os.makedirs(SKY_DIR, exist_ok=True)

# (file_name, lambda_min_nm, lambda_max_nm).  Add entries as you collect maps.
BAND_FILES = [
    ('milky_way_panorama.jpg',         380.0,    700.0),    # Visible (ESO)
    ('2mass_jhk.jpg',                 1140.0,   2310.0),    # 2MASS J/H/K composite
    ('wise_w1w2.jpg',                 3000.0,   5000.0),    # WISE W1+W2
    ('wise_w3.jpg',                   8000.0,  16000.0),    # WISE W3
    ('iras_60um.jpg',                40000.0, 120000.0),    # IRAS 60µm
]

fov_rad = np.deg2rad(FOV_DEG)

bands = []
for fname, lmin, lmax in BAND_FILES:
    path = os.path.join(SKY_DIR, fname)
    # Check Drive root too (cell 8 stores the visible panorama there)
    if not os.path.exists(path):
        alt = os.path.join('/content/drive/MyDrive/WarpDrives/assets', fname)
        if os.path.exists(alt): path = alt
    if os.path.exists(path):
        print(f'  band {lmin:7.0f}–{lmax:7.0f} nm  ←  {path}')
        bands.append((make_image_sky(path, gain=1.6), lmin, lmax))
    else:
        # Synthetic fallback for missing IR maps so the demo still runs
        seed = int((lmin + lmax) / 100)  # deterministic per band
        synth = make_procedural_starfield(
            n_stars=4500, seed=seed, star_radius_px=1.0,
            fov_scale=fov_rad / WIDTH,
            include_milky_way=True,
            background=(0.02, 0.01, 0.005) if lmin > 700 else (0.0, 0.0, 0.015),
        )
        print(f'  band {lmin:7.0f}–{lmax:7.0f} nm  ←  synthetic (no file at {path})')
        bands.append((synth, lmin, lmax))

multiband_sky = make_multiband_sky(bands, smooth_blend=True)

MB_RUN_NAME = RUN_NAME + '_multiband'
MB_RUN_DIR  = os.path.join(DRIVE_ROOT, 'skyflight', MB_RUN_NAME)
MB_FRAMES   = os.path.join(MB_RUN_DIR, 'frames')
MB_VIDEO    = os.path.join(MB_RUN_DIR, 'video')
for d in (MB_RUN_DIR, MB_FRAMES, MB_VIDEO):
    os.makedirs(d, exist_ok=True)

# Multiband demo uses MONOCHROMATIC mode (the band selection IS the
# spectral correction; we don't want the blackbody approximation on
# top of it).
MB_CFG = dataclasses.replace(
    JAX_CFG,
    doppler_mode='monochromatic',
    doppler_intensity_power=3.0,
)

def mb_path(i): return os.path.join(MB_FRAMES, f'frame_{i:04d}.png')

n_pre = sum(1 for i in range(N_FRAMES) if os.path.exists(mb_path(i)))
print(f'{n_pre}/{N_FRAMES} multiband frames already on Drive — resuming.')

rendered = 0
t_session = time.time()
for i, v in enumerate(velocities):
    fp = mb_path(i)
    if os.path.exists(fp): continue
    t0 = time.time()
    img = render_frame_jax(METRIC, dict(BASE_PARAMS, v0=float(v)),
                           multiband_sky, MB_CFG)
    rgb = np.clip(img * 255, 0, 255).astype(np.uint8)
    tmp = fp + '.tmp'
    iio.imwrite(tmp, rgb); os.replace(tmp, fp)
    rendered += 1
    sess = time.time() - t_session
    rate = rendered / max(sess, 1e-6); eta = (N_FRAMES - i - 1) / max(rate, 1e-6)
    print(f'  mb  {i+1}/{N_FRAMES} v={v:.3f} {time.time()-t0:.1f}s ETA {eta/60:.1f} min')

frames = [iio.imread(mb_path(i)) for i in range(N_FRAMES) if os.path.exists(mb_path(i))]
extended = list(frames) + [frames[-1]] * 6 + list(reversed(frames))
MB_MP4 = os.path.join(MB_VIDEO, f'{MB_RUN_NAME}.mp4')
iio.mimsave(MB_MP4, extended, fps=24)
print('Wrote', MB_MP4)
Video(MB_MP4, embed=True, width=512)

## 10. Side-by-side: Alcubierre vs Natário (also Drive-checkpointed)

Same checkpoint pattern: each composite (Alcubierre | Natário) frame
is its own PNG.  Resume by re-running.  Capped at v < 1 because
Natário's central observer is non-timelike at v ≥ 1.

In [ ]:
COMP_RUN_NAME = f'compare_alcubierre_natario_{PRESET}'
COMP_RUN_DIR  = os.path.join(DRIVE_ROOT, 'skyflight', COMP_RUN_NAME)
COMP_FRAMES   = os.path.join(COMP_RUN_DIR, 'frames')
COMP_VIDEO    = os.path.join(COMP_RUN_DIR, 'video')
for d in (COMP_RUN_DIR, COMP_FRAMES, COMP_VIDEO):
    os.makedirs(d, exist_ok=True)

# Comparison renders each side at half-width, so the concatenated frame
# matches the main preset's aspect ratio.
COMP_W     = WIDTH // 2
COMP_H     = HEIGHT
COMP_NF    = max(30, N_FRAMES // 2)
COMP_VMAX  = 0.95   # Natário central observer is timelike only at v < 1
comp_cfg = JaxRenderConfig(
    width=COMP_W, height=COMP_H, fov_deg=FOV_DEG,
    n_steps=JAX_CFG.n_steps, dlam=0.15, chunk_size=8192,
    supersample=JAX_CFG.supersample,
    enable_doppler=True,
    doppler_intensity_power=3.0,
    doppler_tonemap=False,
    enable_horizon_mask=False,
    derivative_mode='auto',     # auto picks FD christoffels for Natário
)
comp_velocities = make_velocities(COMP_NF, COMP_VMAX)

if not USE_REAL_PANORAMA:
    print('WARNING: comparison will use procedural stars; expect twinkling.')
    print('         Run cell 8 (the ESO download) to get the panorama on Drive.')

def comp_path(idx):
    return os.path.join(COMP_FRAMES, f'compare_{idx:04d}.png')

def render_comp_frame(v):
    a = render_frame_jax('alcubierre',
                         dict(v0=float(v), R=1.0, sigma=8.0, shape='tanh'),
                         sky, comp_cfg)
    n = render_frame_jax('natario',
                         dict(v0=float(v), R=1.0, sigma=8.0),
                         sky, comp_cfg)
    return np.concatenate([a, n], axis=1)

for i, v in enumerate(comp_velocities):
    fp = comp_path(i)
    if os.path.exists(fp):
        continue
    t0 = time.time()
    side = render_comp_frame(v)
    rgb = np.clip(side * 255, 0, 255).astype(np.uint8)
    tmp = fp + '.tmp'
    iio.imwrite(tmp, rgb)
    os.replace(tmp, fp)
    print(f'  comp {i+1}/{len(comp_velocities)} v={v:.3f} {time.time()-t0:.1f}s')

comp_frames = [iio.imread(comp_path(i))
               for i in range(len(comp_velocities))
               if os.path.exists(comp_path(i))]
COMP_MP4 = os.path.join(COMP_VIDEO, f'{COMP_RUN_NAME}.mp4')
iio.mimsave(COMP_MP4, comp_frames, fps=18)
print('Wrote', COMP_MP4)
Video(COMP_MP4, embed=True, width=900)

## Resume / restart cookbook

* **Runtime timed out mid-render?**  Re-run the notebook from the top.
  Cells 1–4 are idempotent; cell 5 reads the existing PNGs and only
  renders the missing ones.  The schedule in `meta.json` guarantees
  the same `velocities` list, so a partially-rendered run stays
  internally consistent.
* **Want a different resolution / `v_max` / metric?**  Change
  `RUN_NAME` to a fresh string in cell 1 — that gives you a new
  output directory and `meta.json`, leaving the previous renders
  intact on Drive.
* **Lost the runtime entirely?**  Frames are on Drive.  In a brand
  new runtime, just rerun: cells 1–4 reattach to Drive, cell 5
  picks up where you left off, cell 6 stitches the final video.
* **Crashed mid-write of a PNG?**  We use atomic write
  (`tmp → rename`), so an interrupted write leaves a `.tmp` file
  that the resume logic ignores.

## Notes & tips

- The renderer integrates a **fixed-step RK4**.  ``n_steps × dlam`` should
  exceed the time the slowest ray needs to leave the bubble influence;
  any extra steps are spent in flat space and don't change the
  asymptotic direction.
- ``chunk_size`` controls how many rays are vmapped together.  On a T4,
  4–8k is comfortable; on an A100 you can push it to 16–32k.
- Drive I/O is the slow part of the loop on a fast GPU — at
  ~256×256 the JAX render is ~1s, the `imwrite` to Drive ~0.2–0.5s.
  Don't use a smaller `chunk_size` to "save memory" if it makes
  rendering slower than the I/O.
- The Alcubierre interior observer is *naturally co-moving* — there's
  no SR boost between observer and bubble, so aberration in the
  conventional sense is mild.  The dramatic deformations show up at
  oblique angles where rays pass through more of the bubble wall.